# Классификация марок автомобилей: улучшенный пайплайн обучения и анализа

В этой версии ноутбука добавлены:

- устойчивый запуск на CPU/GPU без падения при отсутствии видеокарты;
- проверка структуры датасета, битых изображений и пересечений между `train`, `val`, `test`;
- анализ дисбаланса классов и расчёт `class_weight`;
- более сильная аугментация;
- метрики `accuracy`, `top-3 accuracy`, `top-5 accuracy`;
- улучшенный classifier head для transfer learning;
- поэтапный fine-tuning EfficientNetB0;
- confusion matrix, classification report и анализ ошибочных предсказаний;
- заготовки для экспериментов с другими backbone-моделями и большим разрешением.

> Важно: ячейки обучения не запускались заново при редактировании ноутбука. Запустите ноутбук сверху вниз на машине, где доступен датасет.

In [ ]:
import os
import random
import hashlib
from pathlib import Path
from collections import Counter

# Уменьшаем объём служебных логов TensorFlow.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.applications.efficientnet import EfficientNetB2, preprocess_input

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, top_k_accuracy_score

print("Версия TensorFlow:", tf.__version__)

# Безопасная настройка GPU: код больше не падает, если GPU нет.
gpus = tf.config.list_physical_devices("GPU")
print("Доступные GPU:", gpus)

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Memory growth для GPU включён.")
    except Exception as exc:
        print("Не удалось настроить memory growth:", exc)
else:
    print("GPU не найден. Ноутбук можно запустить на CPU, но обучение будет медленнее.")

# Mixed precision обычно ускоряет обучение на современных GPU.
# Если появляются нестабильность или NaN, отключите эту строку.
if gpus:
    try:
        tf.keras.mixed_precision.set_global_policy("mixed_float16")
        print("Mixed precision включён:", tf.keras.mixed_precision.global_policy())
    except Exception as exc:
        print("Mixed precision не включён:", exc)

In [ ]:
# Воспроизводимость экспериментов.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Основные параметры. Для тонких визуальных различий между марками можно попробовать (260, 260) или (300, 300).
IMG_SIZE = (256, 256)
BATCH_SIZE = 32
AUTO = tf.data.AUTOTUNE

DATA_DIR = Path("/home/sm/Desktop/ml/CarBrandClassificationDataset")

## 1. Загрузка структуры датасета

Ожидается структура:

```text
CarBrandClassificationDataset/
├── train/
│   ├── class_1/
│   └── ...
├── val/
│   ├── class_1/
│   └── ...
└── test/
    ├── class_1/
    └── ...
```

В таблицы ниже собираются пути к изображениям и строковые метки классов.

In [ ]:
train_base = DATA_DIR / "train"
val_base = DATA_DIR / "val"
test_base = DATA_DIR / "test"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def build_split_df(split_dir: Path) -> pd.DataFrame:
    """Собирает таблицу path/label для одного split'а."""
    if not split_dir.exists():
        raise FileNotFoundError(f"Папка split'а не найдена: {split_dir}")

    rows = []
    class_dirs = sorted([d for d in split_dir.iterdir() if d.is_dir()])
    for cls_dir in class_dirs:
        cls = cls_dir.name
        for f in cls_dir.rglob("*"):
            if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS:
                rows.append((str(f), cls))
    return pd.DataFrame(rows, columns=["path", "label"])

df_train = build_split_df(train_base)
df_val = build_split_df(val_base)
df_test = build_split_df(test_base)

print("Количество изображений:")
print("Train:", len(df_train), "| классов:", df_train["label"].nunique())
print("Val  :", len(df_val), "| классов:", df_val["label"].nunique())
print("Test :", len(df_test), "| классов:", df_test["label"].nunique())

display(pd.concat({
    "train": df_train["label"].value_counts().sort_index(),
    "val": df_val["label"].value_counts().sort_index(),
    "test": df_test["label"].value_counts().sort_index(),
}, axis=1).fillna(0).astype(int).head())

In [ ]:
# Единое отображение классов в индексы.
classes = sorted(df_train["label"].unique().tolist())
class2idx = {c: i for i, c in enumerate(classes)}
idx2class = {i: c for c, i in class2idx.items()}
num_classes = len(classes)

for name, df in [("val", df_val), ("test", df_test)]:
    missing = set(df["label"].unique()) - set(classes)
    if missing:
        raise ValueError(f"В {name} есть классы, которых нет в train: {sorted(missing)}")

df_train["label_idx"] = df_train["label"].map(class2idx)
df_val["label_idx"] = df_val["label"].map(class2idx)
df_test["label_idx"] = df_test["label"].map(class2idx)

print("Число классов:", num_classes)
print("Пример отображения:", list(class2idx.items())[:10])
df_train.head()

## 2. Проверка качества данных

Перед улучшением модели важно проверить сам датасет:

- нет ли сильного дисбаланса классов;
- нет ли пересечений одних и тех же файлов между `train`, `val`, `test`;
- читаются ли изображения без ошибок;
- совпадает ли набор классов между split'ами.

Такая проверка часто даёт больший прирост качества, чем замена архитектуры.

In [ ]:
def plot_class_distribution(df: pd.DataFrame, title: str) -> None:
    counts = df["label"].value_counts().sort_values(ascending=False)
    plt.figure(figsize=(14, 5))
    sns.barplot(x=counts.index, y=counts.values)
    plt.title(title)
    plt.xlabel("Класс")
    plt.ylabel("Количество изображений")
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()

plot_class_distribution(df_train, "Распределение классов в train")
plot_class_distribution(df_val, "Распределение классов в val")
plot_class_distribution(df_test, "Распределение классов в test")

class_counts = df_train["label_idx"].value_counts().sort_index()
imbalance_ratio = class_counts.max() / max(class_counts.min(), 1)
print(f"Коэффициент дисбаланса train max/min: {imbalance_ratio:.2f}")

# class_weight помогает, если часть классов встречается заметно реже.
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_classes),
    y=df_train["label_idx"].values
)
class_weight = {i: float(w) for i, w in enumerate(class_weights_array)}
print("Пример class_weight:", dict(list(class_weight.items())[:10]))

In [ ]:
def file_md5(path: str, chunk_size: int = 1024 * 1024) -> str:
    """Считает MD5 файла. Используется только для диагностики дубликатов."""
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

# Для больших датасетов можно сначала проверить только имена файлов,
# но хеши надёжнее: они находят одинаковые изображения с разными именами.
def add_hashes(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    df = df.copy()
    print(f"Считаю хеши для {split_name}: {len(df)} файлов")
    df["md5"] = [file_md5(p) for p in df["path"]]
    return df

CHECK_DUPLICATES_BY_HASH = True

if CHECK_DUPLICATES_BY_HASH:
    df_train_h = add_hashes(df_train, "train")
    df_val_h = add_hashes(df_val, "val")
    df_test_h = add_hashes(df_test, "test")

    train_hashes = set(df_train_h["md5"])
    val_hashes = set(df_val_h["md5"])
    test_hashes = set(df_test_h["md5"])

    print("Пересечения по хешам:")
    print("train ∩ val :", len(train_hashes & val_hashes))
    print("train ∩ test:", len(train_hashes & test_hashes))
    print("val ∩ test  :", len(val_hashes & test_hashes))

    duplicated_train = df_train_h[df_train_h.duplicated("md5", keep=False)].sort_values("md5")
    print("Дубликаты внутри train:", len(duplicated_train))
else:
    print("Проверка дубликатов по хешам отключена.")

In [ ]:
def validate_images(paths, max_errors: int = 20):
    """Проверяет, что изображения декодируются TensorFlow."""
    bad = []
    for path in paths:
        try:
            raw = tf.io.read_file(path)
            img = tf.image.decode_image(raw, channels=3, expand_animations=False)
            _ = img.shape
        except Exception as exc:
            bad.append((path, str(exc)))
            if len(bad) >= max_errors:
                break
    return bad

bad_train = validate_images(df_train["path"].tolist())
bad_val = validate_images(df_val["path"].tolist())
bad_test = validate_images(df_test["path"].tolist())

print("Битые изображения:")
print("train:", len(bad_train), "| val:", len(bad_val), "| test:", len(bad_test))
if bad_train or bad_val or bad_test:
    display(pd.DataFrame(bad_train + bad_val + bad_test, columns=["path", "error"]).head(20))

## 3. `tf.data`-пайплайн и аугментация

Для baseline-модели используются изображения в диапазоне `[0, 1]`.

Для EfficientNet используется `preprocess_input`, совместимый с ImageNet-предобучением. Аугментация стала сильнее, но без чрезмерных искажений: марку машины нельзя менять агрессивными трансформациями.

In [ ]:
def decode_image(path):
    """Читает изображение, приводит к RGB, float32 и нужному размеру."""
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.convert_image_dtype(img, tf.float32)
    image = tf.image.resize_with_pad(image, IMG_SIZE[0], IMG_SIZE[1])
    return img

augmenter = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(0.08, 0.08),
    layers.RandomContrast(0.15),
], name="augmenter")

def make_ds(paths, labels=None, augment=False, shuffle=False, efficientnet_preprocess=False):
    """Создаёт tf.data.Dataset для обучения или инференса."""
    p = tf.constant(paths)

    if labels is None:
        ds = tf.data.Dataset.from_tensor_slices(p)
        ds = ds.map(lambda x: decode_image(x), num_parallel_calls=AUTO)
    else:
        l = tf.constant(labels, dtype=tf.int32)
        ds = tf.data.Dataset.from_tensor_slices((p, l))
        ds = ds.map(lambda x, y: (decode_image(x), y), num_parallel_calls=AUTO)

    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(paths), 4096), seed=SEED, reshuffle_each_iteration=True)

    if augment and labels is not None:
        ds = ds.map(lambda x, y: (augmenter(x, training=True), y), num_parallel_calls=AUTO)

    if efficientnet_preprocess:
        if labels is None:
            ds = ds.map(lambda x: preprocess_input(x * 255.0), num_parallel_calls=AUTO)
        else:
            ds = ds.map(lambda x, y: (preprocess_input(x * 255.0), y), num_parallel_calls=AUTO)

    ds = ds.batch(BATCH_SIZE).prefetch(AUTO)
    return ds

train_ds = make_ds(
    df_train["path"].tolist(),
    df_train["label_idx"].tolist(),
    augment=True,
    shuffle=True,
    efficientnet_preprocess=False,
)
val_ds = make_ds(df_val["path"].tolist(), df_val["label_idx"].tolist())
test_ds = make_ds(df_test["path"].tolist(), df_test["label_idx"].tolist())

train_ds_tl = make_ds(
    df_train["path"].tolist(),
    df_train["label_idx"].tolist(),
    augment=True,
    shuffle=True,
    efficientnet_preprocess=True,
)
val_ds_tl = make_ds(
    df_val["path"].tolist(),
    df_val["label_idx"].tolist(),
    efficientnet_preprocess=True,
)
test_ds_tl = make_ds(
    df_test["path"].tolist(),
    df_test["label_idx"].tolist(),
    efficientnet_preprocess=True,
)

for bx, by in train_ds_tl.take(1):
    print("Batch:", bx.shape, bx.dtype, "| labels:", by.shape)

## 4. Baseline CNN

Baseline нужен не для максимального качества, а как контрольная точка. Если baseline даёт результат около случайного угадывания, это подтверждает необходимость transfer learning.

In [ ]:
os.makedirs("models", exist_ok=True)

def build_baseline_model(num_classes: int) -> tf.keras.Model:
    inputs = layers.Input(shape=(*IMG_SIZE, 3))

    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dropout(0.35)(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.30)(x)

    logits = layers.Dense(num_classes)(x)
    outputs = layers.Activation("softmax", dtype="float32")(logits)

    model = models.Model(inputs, outputs, name="baseline_cnn")
    model.compile(
        optimizer=optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3_accuracy"),
            tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5_accuracy"),
        ],
    )
    return model

baseline = build_baseline_model(num_classes)
baseline.summary()

In [ ]:
baseline_callbacks = [
    callbacks.ModelCheckpoint(
        "models/baseline_best.keras",
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    callbacks.EarlyStopping(
        monitor="val_accuracy",
        mode="max",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
]

history_baseline = baseline.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=baseline_callbacks,
    class_weight=class_weight,
)

## 5. Transfer learning: EfficientNetB0

По сравнению с исходным вариантом улучшены:

- classifier head: `Dense + BatchNorm + Dropout`;
- метрики top-3/top-5;
- `class_weight` для борьбы с дисбалансом;
- более аккуратный fine-tuning с меньшим learning rate;
- возможность размораживать backbone блоками.

Для дальнейших экспериментов можно заменить `EfficientNetB0` на `EfficientNetB2/B3`, `MobileNetV3Large`, `ResNet50V2`, `ConvNeXtTiny`.

In [ ]:
def build_efficientnet_b2(num_classes: int, img_size=IMG_SIZE, dropout=0.40) -> tuple[tf.keras.Model, tf.keras.Model]:
    """Создаёт модель EfficientNetB0 + улучшенную классификационную голову."""
    base = EfficientNetB2(
        include_top=False,
        weights="imagenet",
        input_shape=(*img_size, 3),
    )
    base.trainable = False

    inputs = layers.Input(shape=(*img_size, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(512, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)

    x = layers.Dense(256, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout * 0.75)(x)

    logits = layers.Dense(num_classes)(x)
    outputs = layers.Activation("softmax", dtype="float32")(logits)

    model = models.Model(inputs, outputs, name="efficientnetb0_car_brand")
    return model, base

def compile_model(model: tf.keras.Model, lr: float) -> None:
    model.compile(
        optimizer=optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3_accuracy"),
            tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5_accuracy"),
        ],
    )

model_tl, base = build_efficientnet_b2(num_classes)
compile_model(model_tl, lr=3e-4)
model_tl.summary()

In [ ]:
tl_callbacks = [
    callbacks.ModelCheckpoint(
        "models/effb0_best.keras",
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    callbacks.EarlyStopping(
        monitor="val_accuracy",
        mode="max",
        patience=6,
        restore_best_weights=True,
        verbose=1,
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1,
    ),
    callbacks.CSVLogger("models/effb0_training_log.csv", append=False),
]

print("Этап 1: обучение classifier head при замороженном backbone.")
hist_warmup = model_tl.fit(
    train_ds_tl,
    validation_data=val_ds_tl,
    epochs=12,
    callbacks=tl_callbacks,
    class_weight=class_weight,
)

## 6. Fine-tuning EfficientNetB0

После warmup размораживаем только верхнюю часть backbone. BatchNorm-слои оставляем замороженными: это обычно делает fine-tuning стабильнее на небольших датасетах.

Если качество растёт медленно, можно попробовать:

- `FINE_TUNE_LAYERS = 80` или `120`;
- learning rate `1e-5` или `3e-6`;
- большее разрешение `IMG_SIZE = (260, 260)`.

In [ ]:
def set_fine_tuning(base_model: tf.keras.Model, fine_tune_layers: int) -> None:
    """Размораживает последние fine_tune_layers слоёв, кроме BatchNormalization."""
    base_model.trainable = True

    for layer in base_model.layers[:-fine_tune_layers]:
        layer.trainable = False

    for layer in base_model.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

FINE_TUNE_LAYERS = 80
set_fine_tuning(base, fine_tune_layers=FINE_TUNE_LAYERS)

trainable_count = sum(int(layer.trainable) for layer in base.layers)
print(f"Trainable слоёв backbone: {trainable_count} из {len(base.layers)}")

compile_model(model_tl, lr=1e-5)

print("Этап 2: fine-tuning верхних слоёв EfficientNetB0.")
hist_ft = model_tl.fit(
    train_ds_tl,
    validation_data=val_ds_tl,
    epochs=40,
    callbacks=tl_callbacks,
    class_weight=class_weight,
)

## 7. Графики обучения

Графики позволяют быстро увидеть:

- есть ли переобучение;
- достигла ли модель плато;
- помог ли fine-tuning после warmup;
- не слишком ли большой learning rate.

In [ ]:
def merge_histories(*histories):
    merged = {}
    for history in histories:
        if history is None:
            continue
        for key, values in history.history.items():
            merged.setdefault(key, []).extend(values)
    return merged

def plot_training_curves(history_dict, title="Обучение модели"):
    metrics = [
        ("accuracy", "val_accuracy", "Accuracy"),
        ("top3_accuracy", "val_top3_accuracy", "Top-3 accuracy"),
        ("top5_accuracy", "val_top5_accuracy", "Top-5 accuracy"),
        ("loss", "val_loss", "Loss"),
    ]

    for train_key, val_key, metric_name in metrics:
        if train_key not in history_dict or val_key not in history_dict:
            continue

        plt.figure(figsize=(8, 5))
        plt.plot(history_dict[train_key], label="train")
        plt.plot(history_dict[val_key], label="val")
        plt.title(f"{title}: {metric_name}")
        plt.xlabel("Эпоха")
        plt.ylabel(metric_name)
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

history_all = merge_histories(hist_warmup, hist_ft)
plot_training_curves(history_all, title="EfficientNetB0")

## 8. Оценка на test и подробные метрики

Обычная accuracy не всегда достаточно информативна. Для 33 классов полезны:

- `top-3 accuracy`: правильная марка попала в 3 наиболее вероятных варианта;
- `top-5 accuracy`: правильная марка попала в 5 вариантов;
- precision/recall/F1 по каждому классу;
- confusion matrix.

In [ ]:
best_tl_path = "models/effb0_best.keras"

if os.path.exists(best_tl_path):
    model_tl.load_weights(best_tl_path)
    print("Загружены лучшие веса:", best_tl_path)
else:
    print("Лучший checkpoint не найден, используются текущие веса модели.")

test_metrics = model_tl.evaluate(test_ds_tl, verbose=0)
print("Метрики EfficientNetB0 на test:")
for name, value in zip(model_tl.metrics_names, test_metrics):
    print(f"{name}: {value:.4f}")

y_true = df_test["label_idx"].values
y_pred_proba = model_tl.predict(test_ds_tl, verbose=1)
y_pred = np.argmax(y_pred_proba, axis=1)

print("\nПроверка top-k через sklearn:")
print("Top-3 accuracy:", top_k_accuracy_score(y_true, y_pred_proba, k=3, labels=np.arange(num_classes)))
print("Top-5 accuracy:", top_k_accuracy_score(y_true, y_pred_proba, k=5, labels=np.arange(num_classes)))

report = classification_report(
    y_true,
    y_pred,
    target_names=classes,
    digits=4,
    zero_division=0,
    output_dict=True,
)
report_df = pd.DataFrame(report).T
display(report_df)

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=np.arange(num_classes))

plt.figure(figsize=(16, 14))
sns.heatmap(
    cm,
    cmap="Blues",
    xticklabels=classes,
    yticklabels=classes,
    square=False,
    cbar=True,
)
plt.title("Confusion matrix на test")
plt.xlabel("Предсказанный класс")
plt.ylabel("Истинный класс")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Нормированная матрица показывает доли ошибок внутри каждого истинного класса.
cm_norm = cm.astype("float") / np.maximum(cm.sum(axis=1, keepdims=True), 1)

plt.figure(figsize=(16, 14))
sns.heatmap(
    cm_norm,
    cmap="Blues",
    xticklabels=classes,
    yticklabels=classes,
    square=False,
    cbar=True,
)
plt.title("Нормированная confusion matrix на test")
plt.xlabel("Предсказанный класс")
plt.ylabel("Истинный класс")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Таблица самых частых пар ошибок.
error_pairs = []
for true_idx in range(num_classes):
    for pred_idx in range(num_classes):
        if true_idx != pred_idx and cm[true_idx, pred_idx] > 0:
            error_pairs.append({
                "true_class": idx2class[true_idx],
                "pred_class": idx2class[pred_idx],
                "count": int(cm[true_idx, pred_idx]),
                "share_within_true_class": float(cm_norm[true_idx, pred_idx]),
            })

error_pairs_df = pd.DataFrame(error_pairs).sort_values(
    ["count", "share_within_true_class"],
    ascending=False,
)
display(error_pairs_df.head(30))

## 9. Анализ ошибочных предсказаний

Ниже выводятся изображения, где модель ошиблась сильнее всего. Это помогает понять источник проблемы:

- визуально похожие марки;
- неправильные labels;
- слишком маленький или закрытый логотип;
- плохое качество изображения;
- доменный сдвиг между train и test.

In [ ]:
def show_misclassified_examples(df, y_true, y_pred, y_pred_proba, n=24):
    wrong_idx = np.where(y_true != y_pred)[0]
    if len(wrong_idx) == 0:
        print("Ошибок не найдено.")
        return

    # Уверенность модели в ошибочном предсказании.
    wrong_conf = y_pred_proba[wrong_idx, y_pred[wrong_idx]]
    order = wrong_idx[np.argsort(-wrong_conf)]
    selected = order[:n]

    cols = 4
    rows = int(np.ceil(len(selected) / cols))
    plt.figure(figsize=(4 * cols, 4 * rows))

    for i, idx in enumerate(selected, start=1):
        img = decode_image(df.iloc[idx]["path"]).numpy()

        true_name = idx2class[int(y_true[idx])]
        pred_name = idx2class[int(y_pred[idx])]
        conf = float(y_pred_proba[idx, y_pred[idx]])

        top5 = np.argsort(-y_pred_proba[idx])[:5]
        top5_text = ", ".join([f"{idx2class[int(j)]}: {y_pred_proba[idx, j]:.2f}" for j in top5])

        plt.subplot(rows, cols, i)
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"true: {true_name}\npred: {pred_name} ({conf:.2f})", fontsize=9)
        print(f"{idx}. true={true_name}, pred={pred_name}, top5=[{top5_text}], path={df.iloc[idx]['path']}")

    plt.tight_layout()
    plt.show()

show_misclassified_examples(df_test, y_true, y_pred, y_pred_proba, n=24)

## 10. Идеи следующих экспериментов

На основе текущего качества около 44% accuracy для EfficientNetB0, наиболее перспективные направления:

1. **Проверить датасет**: дубликаты, ошибки разметки, слишком похожие классы, дисбаланс.
2. **Использовать большее разрешение**: `260×260` или `300×300`, потому что различия между марками часто мелкие.
3. **Попробовать backbone сильнее**: EfficientNetB2/B3, ConvNeXtTiny, ResNet50V2.
4. **Дольше обучать classifier head**: 10–20 эпох до fine-tuning.
5. **Размораживать backbone постепенно**: например 40 → 80 → 120 верхних слоёв с LR `1e-5`, затем `3e-6`.
6. **Использовать MixUp/CutMix**: может повысить устойчивость, особенно при малом датасете.
7. **Смотреть top-5 accuracy**: для задачи из 33 марок top-5 часто показывает, умеет ли модель сужать выбор до похожих вариантов.
8. **Анализировать ошибки по классам**: если отдельные классы почти не распознаются, нужны дополнительные изображения или очистка labels.

Главный практический шаг после запуска этого ноутбука — изучить `classification_report`, confusion matrix и изображения ошибочных предсказаний. Именно они подскажут, что даст больший прирост: данные, архитектура или режим fine-tuning.